# Notebook 02: Data Cleaning & Integration Pipeline
### RetailIQ — Demand Forecasting & Multi-Tool Business Assistant

**Objective:** Implement reproducible data transformations, impute promotional markdown nulls safely, handle economic series missingness, flag returns, and merge sales, features, and stores into a unified modelling dataset.


In [1]:
import sys, os
from pathlib import Path
# Add project root to path for src imports
project_root = str(Path(os.path.abspath('')).resolve())
if not os.path.exists(os.path.join(project_root, 'src')):
    project_root = str(Path(os.path.abspath('')).resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
import pandas as pd

loader = DataLoader()
train_df = loader.load_train()
features_df = loader.load_features()
stores_df = loader.load_stores()

cleaner = DataCleaner()
clean_sales = cleaner.clean_sales(train_df)
clean_feat = cleaner.clean_features(features_df)

print(f"Cleaned Sales: {len(clean_sales):,} rows | Returns Flagged: {(clean_sales['Returns_Flag'] == 1).sum():,}")
print(f"Cleaned Features: {len(clean_feat):,} rows | Missing MarkDowns filled with 0.0")


Cleaned Sales: 421,570 rows | Returns Flagged: 1,285
Cleaned Features: 8,190 rows | Missing MarkDowns filled with 0.0


### Safe Dataset Integration
We perform many-to-one merges on `(Store, Date)` and `Store`, asserting that row counts before and after match exactly.


In [2]:
integrated_df, stats = cleaner.integrate_modelling_dataset(train_df, features_df, stores_df)
print("Integration Validation Report:")
for k, v in stats.items():
    if k != 'null_counts':
        print(f"  {k}: {v}")


Integration Validation Report:
  rows: 421570
  columns: 18
  stores: 45
  departments: 81
  date_min: 2010-02-05
  date_max: 2012-10-26
  returns_count: 1285
  returns_pct: 0.305


**Conclusion:**
- Merges are strictly many-to-one with zero Cartesian expansion or row loss (421,570 rows preserved).
- Missing markdowns represent non-promotional weeks and are filled with $0.00$.
- CPI and Unemployment indicators are forward-filled per store.
